In [8]:
#Import linraries
import pandas as pd
import numpy as np

In [9]:
# --- Project Paths ---

#sales_path = '/workspace/sales.csv'
#inventories_path = '/workspace/inventories.csv'
#satisfaction_path = '/workspace/satisfaction.csv'

# --- Local Paths ---
# NOTA: Para probar el código en tu PC, descomenta las 3 líneas de abajo
# y comenta las 3 de arriba. ANTES DE HACER TU ENTREGA FINAL,
# debes volver a dejarlas como están arriba.
sales_path = 'sales.csv'
inventories_path = 'inventories.csv'
satisfaction_path = 'satisfaction.csv'

# Upload the csv files
df_sales = pd.read_csv(sales_path)
df_inventories = pd.read_csv(inventories_path)
df_satisfaction = pd.read_csv(satisfaction_path)

# Delete null values
df_sales = df_sales.dropna()
df_inventories = df_inventories.dropna()
df_satisfaction = df_satisfaction.dropna()

print("Data is loaded and clean")

Data is loaded and clean


In [10]:
# Total Revenue
df_sales['Total_Revenue'] = df_sales['Cantidad_Vendida'] * df_sales['Precio_Unitario']

# Statistical Summary 
print("--- General Statistical Summary ---")
display(df_sales.describe())

# Total revenue by store
revenue_by_store = df_sales.groupby('ID_Tienda')['Total_Revenue'].sum().reset_index()
print("\n--- Total Revenue by Store ---")
display(revenue_by_store)

# Average revenue by store and product
avg_revenue_prod = df_sales.groupby(['ID_Tienda', 'Producto'])['Total_Revenue'].mean().reset_index()
print("\n--- Average Revenue by Store and Product ---")
display(avg_revenue_prod.head())

# Total sales by store and product 
grouped_sales = df_sales.groupby(['ID_Tienda', 'Producto'])['Cantidad_Vendida'].sum().reset_index()

--- General Statistical Summary ---


,ID_Tienda,Cantidad_Vendida,Precio_Unitario,Total_Revenue
count,10.000000,10.000000,10.000000,10.000000
mean,3.000000,25.000000,190.000000,5050.000000
std,1.490712,9.128709,87.559504,3361.960407
min,1.000000,10.000000,100.000000,1000.000000
25%,2.000000,20.000000,100.000000,2625.000000
50%,3.000000,25.000000,200.000000,3500.000000
75%,4.000000,30.000000,275.000000,7875.000000
max,5.000000,40.000000,300.000000,10500.000000



--- Total Revenue by Store ---


,ID_Tienda,Total_Revenue
0,1,5000
1,2,10500
2,3,9000
3,4,13000
4,5,13000



--- Average Revenue by Store and Product ---


,ID_Tienda,Producto,Total_Revenue
0,1,Producto A,2000.0
1,1,Producto B,3000.0
2,2,Producto A,3000.0
3,2,Producto C,7500.0
4,3,Producto A,1000.0


In [11]:
# ==========================================
# STEP 5: INVENTORY ANALYSIS (PANDAS)
# ==========================================

# 1. Merge inventories dataframe with the grouped sales dataframe
df_inventory_sales = pd.merge(df_inventories, grouped_sales, on=['ID_Tienda', 'Producto'], how='left')

# If a product in inventory wasn't sold, fill the NaN with 0
df_inventory_sales['Cantidad_Vendida'] = df_inventory_sales['Cantidad_Vendida'].fillna(0)

# 2. Calculate Inventory Turnover
# Formula: Total Sales / Available Stock
df_inventory_sales['Inventory_Turnover'] = df_inventory_sales['Cantidad_Vendida'] / df_inventory_sales['Stock_Disponible']

print("--- Inventory Merged with Sales and Turnover ---")
display(df_inventory_sales.head())

# 3. Filter stores with critical inventory levels
# Condition: Sales represent less than 10% of Available Stock
critical_inventory = df_inventory_sales[df_inventory_sales['Cantidad_Vendida'] < (0.10 * df_inventory_sales['Stock_Disponible'])]

print("\n--- Critical Inventory Levels (< 10% sold) ---")
display(critical_inventory)

--- Inventory Merged with Sales and Turnover ---


,ID_Tienda,Producto,Stock_Disponible,Fecha_Actualización,Cantidad_Vendida,Inventory_Turnover
0,1,Producto A,50,2023-01-05,20,0.400000
1,1,Producto B,40,2023-01-06,15,0.375000
2,2,Producto A,60,2023-01-07,30,0.500000
3,2,Producto C,45,2023-01-08,25,0.555556
4,3,Producto A,30,2023-01-09,10,0.333333



--- Critical Inventory Levels (< 10% sold) ---


,ID_Tienda,Producto,Stock_Disponible,Fecha_Actualización,Cantidad_Vendida,Inventory_Turnover


In [12]:
# ==========================================
# STEP 6: CUSTOMER SATISFACTION (PANDAS)
# ==========================================

# 1. Filter stores with low satisfaction (< 60%)
low_satisfaction_stores = df_satisfaction[df_satisfaction['Satisfacción_Promedio'] < 60]

print("--- Stores with Low Satisfaction (< 60%) ---")
display(low_satisfaction_stores)

# 2. Relate satisfaction data to sales performance
# We merge the revenue calculated in Step 4 with the satisfaction scores
sales_vs_satisfaction = pd.merge(revenue_by_store, df_satisfaction, on='ID_Tienda', how='inner')

print("\n--- Sales Performance vs Customer Satisfaction ---")
display(sales_vs_satisfaction)

# 3. Recommendations
print("\n--- Strategic Recommendations ---")
print("1. Investigate operations in stores with satisfaction below 60%.")
print("2. Analyze if low satisfaction correlates with lower Total Revenue.")
print("3. Implement staff training and review stock availability in underperforming branches.")

--- Stores with Low Satisfaction (< 60%) ---


,ID_Tienda,Satisfacción_Promedio,Fecha_Evaluación
4,5,55,2023-01-15



--- Sales Performance vs Customer Satisfaction ---


,ID_Tienda,Total_Revenue,Satisfacción_Promedio,Fecha_Evaluación
0,1,5000,85,2023-01-15
1,2,10500,90,2023-01-15
2,3,9000,70,2023-01-15
3,4,13000,65,2023-01-15
4,5,13000,55,2023-01-15



--- Strategic Recommendations ---
1. Investigate operations in stores with satisfaction below 60%.
2. Analyze if low satisfaction correlates with lower Total Revenue.
3. Implement staff training and review stock availability in underperforming branches.


In [13]:
# ==========================================
# STEP 7: OPERATIONS WITH NUMPY
# ==========================================

# 1. Convert the Pandas column to a NumPy array
# We extract pure numbers for faster mathematical operations
sales_array = df_sales['Total_Revenue'].to_numpy()

# 2. Calculate the median and standard deviation
median_sales = np.median(sales_array)
std_dev_sales = np.std(sales_array)

print("--- NumPy Statistical Operations ---")
print(f"Median of Total Sales: ${median_sales:.2f}")
print(f"Standard Deviation of Total Sales: ${std_dev_sales:.2f}")

# 3. Simulate future sales projections
# Set a seed for reproducibility (Crucial for automatic graders)
np.random.seed(42)

# Generate a random growth factor for each sale 
# Let's simulate a random scenario between a 5% drop (0.95) and a 15% growth (1.15)
growth_factors = np.random.uniform(low=0.95, high=1.15, size=len(sales_array))

# Calculate projected sales by multiplying the arrays
projected_sales_array = sales_array * growth_factors

print("\n--- Future Sales Simulation ---")
print(f"First 5 original sales:  {sales_array[:5]}")
print(f"First 5 projected sales: {projected_sales_array[:5].round(2)}")

--- NumPy Statistical Operations ---
Median of Total Sales: $3500.00
Standard Deviation of Total Sales: $3189.44

--- Future Sales Simulation ---
First 5 original sales:  [2000 3000 3000 7500 1000]
First 5 projected sales: [2049.82 3420.43 3289.2  8022.99  981.2 ]
